## Step 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statistics import stdev
from scipy.stats import t, norm, ttest_ind

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.formula.api import ols
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.tools.tools import add_constant

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, scale
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    RidgeCV, LassoCV, ElasticNetCV
)
import sklearn.linear_model as lm
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
print("All imports OK")

## Step 2 — Load Data & Initial Inspection

In [ ]:
df = pd.read_csv('fullData.csv', encoding='latin-1')
print("Shape:", df.shape)
print()
print(df.dtypes)
print()
df.head()

## Step 3 — Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print("=== Descriptive Statistics ===")
print(df.describe())
print()
print("=== Missing Values ===")
print(df.isnull().sum())

In [ ]:
# Distribution of the target variable: deathRate
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.histplot(pd.to_numeric(df['deathRate'], errors='coerce').dropna(), bins=40, kde=True)
plt.title('Distribution of deathRate')
plt.xlabel('deathRate')

plt.subplot(1, 2, 2)
sns.histplot(np.log(pd.to_numeric(df['deathRate'], errors='coerce').dropna()), bins=40, kde=True)
plt.title('Distribution of log(deathRate)')
plt.xlabel('log(deathRate)')
plt.tight_layout()
plt.savefig('deathRate_distribution.png', dpi=120)
plt.close()
print("Distribution plot saved.")

In [ ]:
# Correlation heatmap of numeric features
numeric_preview = df.apply(pd.to_numeric, errors='coerce').dropna(axis=1, how='all')
plt.figure(figsize=(10, 8))
corr = numeric_preview.corr()
sns.heatmap(corr, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120)
plt.close()
print("Correlation heatmap saved.")
print()
print("Correlation with deathRate (top 10):")
print(corr['deathRate'].abs().sort_values(ascending=False).head(11))

In [ ]:
# Scatter plots: key predictors vs deathRate
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
df_num = df.apply(pd.to_numeric, errors='coerce')
pairs = [
    ('incidenceRate', 'deathRate'),
    ('povertyPercent', 'deathRate'),
    ('medIncome',      'deathRate'),
    ('popEst2015',     'deathRate'),
    ('studyCount',     'deathRate'),
]
for ax, (x, y) in zip(axes.flat, pairs):
    ax.scatter(df_num[x], df_num[y], alpha=0.4, s=15)
    ax.set_xlabel(x); ax.set_ylabel(y)
    ax.set_title(f'{x} vs {y}')
axes.flat[-1].axis('off')
plt.suptitle('Key Predictors vs Death Rate', fontsize=14)
plt.tight_layout()
plt.savefig('scatter_plots.png', dpi=120)
plt.close()
print("Scatter plots saved.")

In [ ]:
# Boxplot: deathRate by recentTrend category
df_num2 = df.apply(pd.to_numeric, errors='coerce')
plt.figure(figsize=(8, 5))
trend_labels = df['recentTrend'].unique()
groups = [df_num2.loc[df['recentTrend'] == t, 'deathRate'].dropna() for t in ['falling','stable','rising'] if t in trend_labels]
trend_names = [t for t in ['falling','stable','rising'] if t in trend_labels]
plt.boxplot(groups, labels=trend_names)
plt.title('Death Rate by Recent Trend')
plt.ylabel('deathRate')
plt.tight_layout()
plt.savefig('boxplot_trend.png', dpi=120)
plt.close()
print("Boxplot saved.")

## Step 4 — Hypothesis Testing

In [ ]:
# t-test: high poverty vs low poverty counties
df_num3 = df.apply(pd.to_numeric, errors='coerce').dropna(subset=['povertyPercent','deathRate'])
median_pov = df_num3['povertyPercent'].median()
high_pov = df_num3.loc[df_num3['povertyPercent'] >= median_pov, 'deathRate']
low_pov  = df_num3.loc[df_num3['povertyPercent'] <  median_pov, 'deathRate']

t_stat, p_val = ttest_ind(high_pov, low_pov)
print("=== T-Test: High vs Low Poverty Counties (deathRate) ===")
print(f"High poverty mean deathRate : {high_pov.mean():.2f}")
print(f"Low  poverty mean deathRate : {low_pov.mean():.2f}")
print(f"t-statistic : {t_stat:.4f}")
print(f"p-value     : {p_val:.6f}")
if p_val < 0.05:
    print("Conclusion: Statistically significant difference (p < 0.05)")
else:
    print("Conclusion: No statistically significant difference")

In [ ]:
# One-Way ANOVA: deathRate across recentTrend groups
trend_map_anova = {'falling': 0, 'stable': 1, 'rising': 2}
df_anova = df.copy()
df_anova['recentTrend_label'] = df_anova['recentTrend']
df_anova['recentTrend'] = df_anova['recentTrend'].map(trend_map_anova)
df_anova['deathRate'] = pd.to_numeric(df_anova['deathRate'], errors='coerce')
df_anova.dropna(subset=['deathRate','recentTrend_label'], inplace=True)

groups_anova = [grp['deathRate'].values for _, grp in df_anova.groupby('recentTrend_label')]
f_stat, p_anova = stats.f_oneway(*groups_anova)
print("=== One-Way ANOVA: deathRate across recentTrend groups ===")
print(f"F-statistic : {f_stat:.4f}")
print(f"p-value     : {p_anova:.6f}")
if p_anova < 0.05:
    print("Conclusion: At least one group mean is significantly different (p < 0.05)")
else:
    print("Conclusion: No significant difference between group means")
print()

# Tukey HSD post-hoc
tukey = pairwise_tukeyhsd(df_anova['deathRate'], df_anova['recentTrend_label'])
print("=== Tukey HSD Post-Hoc Test ===")
print(tukey)

## Step 5 — Data Cleaning & Encoding

In [ ]:
# Re-load clean copy for modelling
df = pd.read_csv('fullData.csv', encoding='latin-1')

trend_map = {'falling': 0, 'stable': 1, 'rising': 2}
df['recentTrend']   = df['recentTrend'].map(trend_map).fillna(1)
df['recTrend']      = df['recTrend'].map(trend_map).fillna(1)
df['fiveYearTrend'] = pd.to_numeric(df['fiveYearTrend'], errors='coerce').fillna(0)

numeric_cols = ['povertyPercent', 'medIncome', 'popEst2015', 'incidenceRate', 'studyCount']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
df.dropna(subset=['deathRate'], inplace=True)
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

print(f"Null values in deathRate : {df['deathRate'].isna().sum()}")
print(f"Shape after cleaning     : {df.shape}")

## Step 6 — Log-Transform & Feature Engineering

In [ ]:
df_with_state = df.copy()
df.drop('State', axis=1, inplace=True)

# Log-transform skewed predictors
skewed_cols = ['popEst2015', 'studyCount', 'medIncome', 'incidenceRate']
df_log = df.copy()
for col in skewed_cols:
    df_log[col] = np.log1p(df_log[col])

# Log-transform the target (improves linearity)
df_log['deathRate_log'] = np.log(df_log['deathRate'])

print("Skewness after log transform:")
print(df_log[skewed_cols + ['deathRate_log']].skew().round(3))

In [ ]:
# Interaction & polynomial features (on log-scaled columns)
df_feat = df_log.copy()

df_feat['poverty_X_incidence'] = df_feat['povertyPercent'] * df_feat['incidenceRate']
df_feat['income_X_poverty']    = df_feat['medIncome']      * df_feat['povertyPercent']
df_feat['study_X_incidence']   = df_feat['studyCount']     * df_feat['incidenceRate']
df_feat['income_X_incidence']  = df_feat['medIncome']      * df_feat['incidenceRate']
df_feat['pop_X_poverty']       = df_feat['popEst2015']     * df_feat['povertyPercent']
df_feat['trend_X_incidence']   = df_feat['recentTrend']    * df_feat['incidenceRate']
df_feat['fiveTrend_X_income']  = df_feat['fiveYearTrend']  * df_feat['medIncome']
df_feat['poverty_sq']          = df_feat['povertyPercent'] ** 2
df_feat['incidence_sq']        = df_feat['incidenceRate']  ** 2
df_feat['income_sq']           = df_feat['medIncome']      ** 2

feature_cols = [
    'povertyPercent', 'medIncome', 'popEst2015', 'incidenceRate', 'studyCount',
    'recentTrend', 'recTrend', 'fiveYearTrend',
    'poverty_X_incidence', 'income_X_poverty', 'study_X_incidence',
    'income_X_incidence', 'pop_X_poverty', 'trend_X_incidence', 'fiveTrend_X_income',
    'poverty_sq', 'incidence_sq', 'income_sq'
]
print(f"Feature matrix: {df_feat[feature_cols].shape}")
print("Sample feature stats:")
print(df_feat[feature_cols].describe().loc[['mean','std']].round(3))

## Step 7 — Variance Inflation Factor (Multicollinearity Check)

In [ ]:
X_vif        = df_feat[['povertyPercent', 'medIncome', 'popEst2015', 'incidenceRate', 'studyCount']]
X_with_const = add_constant(X_vif.dropna())
vif_data = pd.DataFrame({
    'Feature': X_with_const.columns,
    'VIF':     [variance_inflation_factor(X_with_const.values, i)
                for i in range(X_with_const.shape[1])]
})
print("VIF (multicollinearity check):")
print(vif_data[vif_data['Feature'] != 'const'].to_string(index=False))

## Step 8 — Define X & Y, Train/Test Split

In [ ]:
Y = df_feat['deathRate_log']
X = df_feat[feature_cols]
print(f"X shape: {X.shape}, Y shape: {Y.shape}")
print(f"Y (log deathRate) range: {Y.min():.3f} to {Y.max():.3f}")

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=123
)
# Keep original (non-log) death rates for final RMSE in original scale
Y_test_orig  = np.exp(Y_test)
Y_train_orig = np.exp(Y_train)
print(f"Train: {X_train.shape}   Test: {X_test.shape}")

## Step 9 — Feature Scaling (StandardScaler)

In [ ]:
scaler_std = StandardScaler()
X_train_std = pd.DataFrame(
    scaler_std.fit_transform(X_train),
    columns=X_train.columns, index=X_train.index
)
X_test_std = pd.DataFrame(
    scaler_std.transform(X_test),
    columns=X_test.columns, index=X_test.index
)
print("StandardScaler applied (fit on train only)")
print(f"X_train_std shape: {X_train_std.shape}")

# Also prepare MinMaxScaler version (used in hackathon)
scaler_mm = MinMaxScaler()
X_train_mm = pd.DataFrame(
    scaler_mm.fit_transform(X_train),
    columns=X_train.columns, index=X_train.index
)
X_test_mm = pd.DataFrame(
    scaler_mm.transform(X_test),
    columns=X_test.columns, index=X_test.index
)
print("MinMaxScaler version also prepared.")

## Step 10 — Baseline OLS Regression

In [ ]:
df_train_ols = pd.concat([X_train_std, Y_train.rename('deathRate_log')], axis=1)
formula = 'deathRate_log ~ ' + ' + '.join(feature_cols)
fit_base = smf.ols(formula, data=df_train_ols).fit()

pred_test_base_log = fit_base.predict(X_test_std)
pred_test_base     = np.exp(pred_test_base_log)

r2_base  = r2_score(Y_test_orig, pred_test_base)
mse_base = mean_squared_error(Y_test_orig, pred_test_base)
print(f"Baseline OLS — Test RMSE: {np.sqrt(mse_base):.4f}  |  Test R²: {r2_base:.4f}")
print()
print(fit_base.summary().tables[1])

## Step 11 — K-Fold Cross-Validation (Linear Regression)

In [ ]:
model_lr = lm.LinearRegression()
K = 10
Kfold = KFold(K, random_state=0, shuffle=True)
mse_cv = cross_val_score(model_lr, X, Y, cv=Kfold, scoring='neg_mean_squared_error')
print("10-Fold CV MSE per fold (log scale):")
print(-mse_cv.round(4))
print(f"Mean CV MSE : {(-mse_cv).mean():.4f}  ±  {(-mse_cv).std():.4f}")

## Step 12 — Ridge Regularization Path

In [ ]:
alphas_path = 10 ** np.linspace(10, -5, 100) * 0.5
coefs_ridge = []
for a in alphas_path:
    ridge_tmp = Ridge(alpha=a)
    ridge_tmp.fit(X_train_std, Y_train)
    coefs_ridge.append(ridge_tmp.coef_)

ax = plt.gca()
ax.plot(alphas_path, coefs_ridge)
ax.set_xscale('log')
plt.axis('tight')
plt.xlabel('alpha')
plt.ylabel('weights')
plt.title('Ridge Coefficients as a Function of Regularization')
plt.tight_layout()
plt.savefig('ridge_coef_path.png', dpi=120)
plt.close()
print("Ridge coefficient path saved.")

## Step 13 — Lasso Regularization Path

In [ ]:
coefs_lasso = []
for a in alphas_path:
    lasso_tmp = Lasso(alpha=a, max_iter=10000)
    lasso_tmp.fit(X_train_std, Y_train)
    coefs_lasso.append(lasso_tmp.coef_)

ax = plt.gca()
ax.plot(alphas_path, coefs_lasso)
ax.set_xscale('log')
plt.axis('tight')
plt.xlabel('alpha')
plt.ylabel('weights')
plt.title('Lasso Coefficients as a Function of Regularization')
plt.tight_layout()
plt.savefig('lasso_coef_path.png', dpi=120)
plt.close()
print("Lasso coefficient path saved.")

## Step 14 — ElasticNet Regularization Path

In [ ]:
coefs_enet = []
for a in alphas_path:
    en_tmp = ElasticNet(alpha=a, max_iter=10000)
    en_tmp.fit(X_train_std, Y_train)
    coefs_enet.append(en_tmp.coef_)

ax = plt.gca()
ax.plot(alphas_path, coefs_enet)
ax.set_xscale('log')
plt.axis('tight')
plt.xlabel('alpha')
plt.ylabel('weights')
plt.title('ElasticNet Coefficients as a Function of Regularization')
plt.tight_layout()
plt.savefig('enet_coef_path.png', dpi=120)
plt.close()
print("ElasticNet coefficient path saved.")

## Step 15 — Combined Coefficient Path Plot (Ridge + Lasso)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(alphas_path, coefs_ridge)
axes[0].set_xscale('log')
axes[0].set_xlabel('Alpha'); axes[0].set_ylabel('Coefficients')
axes[0].set_title('Ridge - Coefficient Path')

axes[1].plot(alphas_path, coefs_lasso)
axes[1].set_xscale('log')
axes[1].set_xlabel('Alpha'); axes[1].set_ylabel('Coefficients')
axes[1].set_title('Lasso - Coefficient Path')

plt.tight_layout()
plt.savefig('coefficient_paths_combined.png', dpi=120)
plt.close()
print("Combined coefficient path plot saved.")

## Step 16 — Ridge Regression with CV Alpha Selection

In [ ]:
alphas = 10 ** np.linspace(6, -4, 200)

ridgecv = RidgeCV(alphas=alphas, cv=5, scoring='r2')
ridgecv.fit(X_train_std, Y_train)
best_alpha_ridge = ridgecv.alpha_
print(f"Best Ridge alpha (CV): {best_alpha_ridge:.6f}")

fitR = Ridge(alpha=best_alpha_ridge)
fitR.fit(X_train_std, Y_train)
pred_ridge_log = fitR.predict(X_test_std)
pred_ridge     = np.exp(pred_ridge_log)
r2_ridge       = r2_score(Y_test_orig, pred_ridge)
rmse_ridge     = np.sqrt(mean_squared_error(Y_test_orig, pred_ridge))
print(f"Ridge — Test RMSE: {rmse_ridge:.4f}  |  Test R²: {r2_ridge:.4f}")

print("\nTop 10 Ridge coefficients:")
print(pd.Series(fitR.coef_, index=X_train_std.columns).abs().sort_values(ascending=False).head(10).round(4))

## Step 17 — Lasso Regression with CV Alpha Selection

In [ ]:
lassocv = LassoCV(alphas=alphas, cv=5, max_iter=50000)
lassocv.fit(X_train_std, Y_train)
best_alpha_lasso = lassocv.alpha_
print(f"Best Lasso alpha (CV): {best_alpha_lasso:.8f}")

fitL = Lasso(alpha=best_alpha_lasso, max_iter=50000)
fitL.fit(X_train_std, Y_train)
pred_lasso_log = fitL.predict(X_test_std)
pred_lasso     = np.exp(pred_lasso_log)
r2_lasso       = r2_score(Y_test_orig, pred_lasso)
rmse_lasso     = np.sqrt(mean_squared_error(Y_test_orig, pred_lasso))
print(f"Lasso — Test RMSE: {rmse_lasso:.4f}  |  Test R²: {r2_lasso:.4f}")

non_zero = np.sum(fitL.coef_ != 0)
print(f"Lasso non-zero coefficients: {non_zero} / {X_train_std.shape[1]}")

## Step 18 — ElasticNet with CV Alpha & l1_ratio Selection

In [ ]:
elasticnetcv = ElasticNetCV(
    alphas=alphas,
    l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 1.0],
    cv=5, max_iter=50000
)
elasticnetcv.fit(X_train_std, Y_train)
best_alpha_enet = elasticnetcv.alpha_
best_l1_ratio   = elasticnetcv.l1_ratio_
print(f"Best ElasticNet alpha: {best_alpha_enet:.8f}  |  l1_ratio: {best_l1_ratio:.2f}")

fitE = ElasticNet(alpha=best_alpha_enet, l1_ratio=best_l1_ratio, max_iter=50000)
fitE.fit(X_train_std, Y_train)
pred_enet_log = fitE.predict(X_test_std)
pred_enet     = np.exp(pred_enet_log)
r2_enet       = r2_score(Y_test_orig, pred_enet)
rmse_enet     = np.sqrt(mean_squared_error(Y_test_orig, pred_enet))
print(f"ElasticNet — Test RMSE: {rmse_enet:.4f}  |  Test R²: {r2_enet:.4f}")

# Predictions in original scale on ALL training data (like hackathon)
Y_all_orig = np.exp(Y)
X_all_std = pd.DataFrame(scaler_std.transform(X), columns=X.columns)
pred_enet_all  = np.exp(fitE.predict(X_all_std))
pred_lr_all    = np.exp(model_lr.fit(X_train_std, Y_train).predict(X_all_std))
r2_enet_all    = r2_score(Y_all_orig, pred_enet_all)
r2_lr_all      = r2_score(Y_all_orig, pred_lr_all)
mse_enet_log   = mean_squared_error(Y, fitE.predict(X_all_std))
print(f"\nR² on full data (ElasticNet)        : {r2_enet_all:.4f}")
print(f"R² on full data (Linear Regression) : {r2_lr_all:.4f}")
print(f"MSE on log-transformed values       : {mse_enet_log:.4f}")

## Step 19 — Polynomial OLS Regression

In [ ]:
df_scaled_train = X_train_std.copy()
df_scaled_train['deathRate_log'] = Y_train.values

for feat, s2, s3 in [('povertyPercent','ppt2','ppt3'),
                      ('medIncome','mi2','mi3'),
                      ('popEst2015','pe2','pe3'),
                      ('incidenceRate','ir2','ir3'),
                      ('studyCount','sc2','sc3')]:
    df_scaled_train[s2] = df_scaled_train[feat] ** 2
    df_scaled_train[s3] = df_scaled_train[feat] ** 3

lm_full = smf.ols(
    'deathRate_log ~ povertyPercent + ppt2 + ppt3'
    '+ medIncome + mi2 + mi3'
    '+ popEst2015 + pe2 + pe3'
    '+ incidenceRate + ir2 + ir3'
    '+ studyCount + sc2 + sc3',
    data=df_scaled_train
).fit()
print(f"Polynomial OLS — Train R²: {lm_full.rsquared:.4f}")

In [ ]:
df_scaled_test = X_test_std.copy()
for feat, s2, s3 in [('povertyPercent','ppt2','ppt3'),
                      ('medIncome','mi2','mi3'),
                      ('popEst2015','pe2','pe3'),
                      ('incidenceRate','ir2','ir3'),
                      ('studyCount','sc2','sc3')]:
    df_scaled_test[s2] = df_scaled_test[feat] ** 2
    df_scaled_test[s3] = df_scaled_test[feat] ** 3

pred_poly_log = lm_full.predict(df_scaled_test)
pred_poly     = np.exp(pred_poly_log)
r2_poly       = r2_score(Y_test_orig, pred_poly)
rmse_poly     = np.sqrt(mean_squared_error(Y_test_orig, pred_poly))
print(f"OLS Polynomial (scaled) — Test RMSE: {rmse_poly:.4f}  |  Test R²: {r2_poly:.4f}")

## Step 20 — Gradient Boosting & Random Forest

In [ ]:
# Gradient Boosting Regressor
gbr = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    min_samples_leaf=5,
    random_state=42
)
gbr.fit(X_train_std, Y_train)
pred_gbr_log = gbr.predict(X_test_std)
pred_gbr     = np.exp(pred_gbr_log)
r2_gbr       = r2_score(Y_test_orig, pred_gbr)
rmse_gbr     = np.sqrt(mean_squared_error(Y_test_orig, pred_gbr))
print(f"GradientBoosting — Test RMSE: {rmse_gbr:.4f}  |  Test R²: {r2_gbr:.4f}")

# Random Forest
rf = RandomForestRegressor(
    n_estimators=300, max_depth=8, min_samples_leaf=3,
    random_state=42, n_jobs=-1
)
rf.fit(X_train_std, Y_train)
pred_rf_log = rf.predict(X_test_std)
pred_rf     = np.exp(pred_rf_log)
r2_rf       = r2_score(Y_test_orig, pred_rf)
rmse_rf     = np.sqrt(mean_squared_error(Y_test_orig, pred_rf))
print(f"RandomForest     — Test RMSE: {rmse_rf:.4f}  |  Test R²: {r2_rf:.4f}")

# Feature importances
feat_imp = pd.Series(gbr.feature_importances_, index=X_train_std.columns)
print("\nTop 10 features (GBR importance):")
print(feat_imp.sort_values(ascending=False).head(10).round(4))

## Step 21 — K-Fold Cross-Validation on Gradient Boosting

In [ ]:
kfold   = KFold(10, random_state=0, shuffle=True)
cv_r2   = cross_val_score(gbr, X_train_std, Y_train, cv=kfold, scoring='r2')
cv_rmse = np.sqrt(-cross_val_score(gbr, X_train_std, Y_train, cv=kfold,
                                    scoring='neg_mean_squared_error'))

print(f"  R2   per fold: {np.round(cv_r2, 4)}")
print(f"  Mean R2: {cv_r2.mean():.4f} +/- {cv_r2.std():.4f}")
print(f"  Mean RMSE (log scale): {cv_rmse.mean():.4f} +/- {cv_rmse.std():.4f}")

## Step 22 — Model Comparison Summary

In [ ]:
results = pd.DataFrame({
    'Model': [
        'Baseline OLS',
        'OLS + Polynomial (scaled)',
        f'Ridge (alpha={best_alpha_ridge:.4f})',
        f'Lasso (alpha={best_alpha_lasso:.6f})',
        f'ElasticNet (alpha={best_alpha_enet:.6f})',
        'Gradient Boosting',
        'Random Forest',
    ],
    'Test RMSE': [
        np.sqrt(mse_base), rmse_poly,
        rmse_ridge, rmse_lasso, rmse_enet,
        rmse_gbr, rmse_rf,
    ],
    'Test R2': [
        r2_base, r2_poly,
        r2_ridge, r2_lasso, r2_enet,
        r2_gbr, r2_rf,
    ]
}).set_index('Model').round(4)

print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)
print(results.to_string())

best_model = results['Test R2'].idxmax()
best_r2    = results['Test R2'].max()
print(f"\nBest model: {best_model}  |  R2 = {best_r2:.4f} ({best_r2*100:.1f}%)")
if best_r2 >= 0.80:
    print("Target of R2 >= 0.80 ACHIEVED!")
else:
    print(f"Current best R2 = {best_r2:.4f}.")

## Step 23 — Residual Diagnostic Plots (Best Model)

In [ ]:
# Use GBR as best model for diagnostics
residuals = Y_test_orig.values - pred_gbr
fitted    = pred_gbr

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Residuals vs Fitted
axes[0].scatter(fitted, residuals, alpha=0.5, s=20)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Fitted values'); axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

# Histogram of residuals
axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].set_xlabel('Residuals'); axes[1].set_title('Residual Distribution')

# Q-Q plot
stats.probplot(residuals, dist="norm", plot=axes[2])
axes[2].set_title('Q-Q Plot of Residuals')

plt.suptitle('GBR Residual Diagnostics', fontsize=13)
plt.tight_layout()
plt.savefig('residual_diagnostics.png', dpi=120)
plt.close()
print("Residual diagnostic plots saved.")

## Step 24 — Feature Importance Bar Chart

In [ ]:
top_n = feat_imp.sort_values(ascending=False).head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=top_n.values, y=top_n.index, palette='viridis')
plt.title('Top 10 Feature Importances (GBR)')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120)
plt.close()
print("Feature importance chart saved.")

## Step 25 — Final Summary

In [ ]:
print("=" * 60)
print("SSDI PROJECT — PIPELINE COMPLETE")
print("=" * 60)
print(f"Dataset shape            : {df_feat.shape}")
print(f"Features used            : {len(feature_cols)}")
print(f"Train / Test split       : 80% / 20%  (random_state=123)")
print()
print("Models evaluated:")
print(results.to_string())
print()
print(f"Best model               : {best_model}")
print(f"Best Test R²             : {best_r2:.4f}  ({best_r2*100:.1f}%)")
print()
print("Saved artefacts:")
for f in ['deathRate_distribution.png', 'correlation_heatmap.png',
          'scatter_plots.png', 'boxplot_trend.png',
          'ridge_coef_path.png', 'lasso_coef_path.png',
          'enet_coef_path.png', 'coefficient_paths_combined.png',
          'residual_diagnostics.png', 'feature_importance.png']:
    print(f"  {f}")